# Foundations 1 — Endpoints, auth, and the three URL paths

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

**Start here.** Every other notebook in this collection assumes what this one
establishes. The four things worth knowing before you write a line of code:

1. **`bedrock-runtime` is the endpoint AWS recommends for new applications**, and
   since August 2026 it speaks all five APIs: InvokeModel, Converse, OpenAI
   Chat Completions, OpenAI Responses, and Anthropic Messages.
2. `bedrock-mantle` is a **second endpoint**, still fully supported, and it is
   where server-side tool use, `background=true`, Projects and Workspaces live.
3. **Neither endpoint's OpenAI-compatible APIs go through boto3.** You call them
   on URL paths. For those, the AWS SDK is useful as a *request signer* — or you
   use a Bedrock API key, which both endpoints accept.
4. A model's **URL path and even its model ID depend on which endpoint you use**.
   Guessing wrong gives a 400, a 404 — or, on `bedrock-runtime`, a **200 that
   means failure**. §2b is about that last one.

## What this notebook covers
- Endpoint choice: what each one serves, and what AWS recommends
- Auth A: SigV4 with botocore · Auth B: short-term API key · Auth C: curl
- The path families on **both** endpoints, demonstrated against live models
- The `UnknownOperationException` that arrives as HTTP 200
- Model discovery, the ID differences between endpoints, the regional footprint
- The IAM permissions you actually need

## Prerequisites
```bash
pip install -r ../requirements.txt
```

Needs openai, boto3, aws-bedrock-token-generator.

`requirements.txt` pins the exact versions this collection was tested
against. An unpinned install resolves whatever is current, which may be
untested or compromised (OWASP LLM03, Supply Chain).

You also need AWS credentials in your environment (profile, role, or instance
metadata) with Bedrock Mantle access.

### Where the helpers come from

The next cell does this:

```python
sys.path.insert(0, "../_shared")
from bedrock import ...
```

`bedrock` is **not** a package from PyPI — it is this collection's own helper
module, [`_shared/bedrock.py`](../_shared/bedrock.py). Every notebook sits one
level down, so `../_shared` puts it on the import path. It exists only to remove
repetition; the notebooks are the teaching material, and nothing in the module is
required to call Bedrock yourself.

What this notebook uses from it:

| Helper | What it does |
|---|---|
| `response_text` | assistant text from a Responses API payload — the raw body has **no** `output_text` field |
| `safe_print` | `print()` with account IDs, IAM principals and opaque service IDs redacted |

Two behaviours worth knowing before you read any output below, because several
cells depend on them:

- **`post()` and `converse()` never raise on a service error.** They return the
  status and body so a cell can *show* a 400 rather than stopping the notebook.
  Many cells here deliberately provoke an error to demonstrate a limit.
- **Anything printed from a control-plane response goes through redaction**, since
  this output is committed to a public repository.


In [1]:
import json
import os
import re
import sys
import urllib.error
import urllib.request

import boto3

sys.path.insert(0, "../_shared")
from bedrock import response_text, safe_print

REGION = "us-east-1"
HOST = f"https://bedrock-mantle.{REGION}.api.aws"

print("boto3", boto3.__version__)
# safe_print redacts the account ID and principal name: this output is
# committed to a public repository (see _shared/bedrock.py).
safe_print("caller:", boto3.client("sts").get_caller_identity()["Arn"])
print("host:  ", HOST)

boto3 1.43.59


caller: arn:aws:iam::123456789012:user/sample-user
host:   https://bedrock-mantle.us-east-1.api.aws


## 1. Two endpoints, and why

Both endpoints can serve the *same* model at the *same* per-token price — AWS
states this explicitly: *"Per-token pricing for the same model is identical on
`bedrock-runtime` and `bedrock-mantle`. Choose an endpoint based on the APIs and
capabilities you need, not cost."* So this section is about capability, and the
table is built from what the endpoints answered today.

The one row that used to decide everything — which APIs each endpoint speaks —
changed in August 2026. `bedrock-runtime` picked up Chat Completions, Responses
and Messages, which is why AWS now recommends it as the default starting point.

In [2]:
# Rows marked (probed) are measured below in this cell rather than remembered.
# The API row is exactly the one that went stale, so it gets measured.
import urllib.error
import urllib.request

from aws_bedrock_token_generator import provide_token

RUNTIME_HOST = f"https://bedrock-runtime.{REGION}.amazonaws.com"


def _served(base: str, path: str, payload: dict, extra: dict | None = None) -> bool:
    """True when this endpoint really serves this path.

    Not `status == 200`: bedrock-runtime answers an unrecognised path with 200 and
    a Coral UnknownOperationException in the body. §2b demonstrates it.
    """
    headers = {
        "Authorization": f"Bearer {provide_token(region=REGION)}",
        "Content-Type": "application/json",
    }
    if extra:
        headers.update(extra)
    req = urllib.request.Request(
        base + path, data=json.dumps(payload).encode(), headers=headers, method="POST"
    )
    try:
        # URL is a literal host constant plus a literal path, so the scheme is
        # https by construction and urllib's other schemes are unreachable.
        # nosemgrep: dynamic-urllib-use-detected
        with urllib.request.urlopen(req, timeout=120) as resp:  # nosec B310  # noqa: S310
            body = resp.read().decode("utf-8", "replace")
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8", "replace")
        # A model-level complaint proves the ROUTE exists; an unknown operation
        # proves it does not. That distinction is the whole point of this probe.
        return "UnknownOperation" not in body and exc.code != 404
    except Exception:
        return False
    return "UnknownOperation" not in body


# One request per API per endpoint, each with a model that endpoint accepts.
CHAT_MT = {"model": "openai.gpt-oss-120b", "messages": [{"role": "user", "content": "Hi"}], "max_tokens": 16}
CHAT_RT = {"model": "openai.gpt-oss-120b-1:0", "messages": [{"role": "user", "content": "Hi"}], "max_completion_tokens": 16}
RESP_MT = {"model": "openai.gpt-oss-120b", "input": "Hi", "max_output_tokens": 16}
RESP_RT = {"model": "us.openai.gpt-5.6-sol", "input": "Hi", "max_output_tokens": 16}
MSG_MT = {"model": "anthropic.claude-opus-5", "max_tokens": 16, "messages": [{"role": "user", "content": "Hi"}]}
MSG_RT = {"model": "us.anthropic.claude-opus-5", "max_tokens": 16, "messages": [{"role": "user", "content": "Hi"}]}
AV = {"anthropic-version": "2023-06-01"}

api_rows = [
    ("Chat Completions", _served(RUNTIME_HOST, "/openai/v1/chat/completions", CHAT_RT),
     _served(HOST, "/v1/chat/completions", CHAT_MT)),
    ("Responses", _served(RUNTIME_HOST, "/openai/v1/responses", RESP_RT),
     _served(HOST, "/v1/responses", RESP_MT)),
    ("Anthropic Messages", _served(RUNTIME_HOST, "/anthropic/v1/messages", MSG_RT, AV),
     _served(HOST, "/anthropic/v1/messages", MSG_MT, AV)),
]
mark = {True: "yes", False: "no"}

comparison = [
    ("URL", "bedrock-runtime.{r}.amazonaws.com", "bedrock-mantle.{r}.api.aws"),
    ("AWS recommends", "yes, for new applications", "fully supported"),
    ("InvokeModel / Converse", "yes", "no"),
]
comparison += [
    (f"{name} (probed)", mark[on_runtime], mark[on_mantle])
    for name, on_runtime, on_mantle in api_rows
]
comparison += [
    ("OpenAI-compatible path", "/openai/v1", "/openai/v1 or /v1, by family"),
    ("Auth", "SigV4 or Bedrock API key", "SigV4 or Bedrock API key"),
    ("OpenAI SDK drop-in", "yes (base_url + key)", "yes (base_url + key)"),
    ("Stateful chat", "store + previous_response_id", "store + previous_response_id"),
    ("Server-side tools", "no", "yes (incl. web search)"),
    ("Async (background=true)", "no (400)", "yes"),
    ("Cross-region (CRIS)", "geo/global/in profiles", "in-Region only"),
    ("App inference profiles", "Converse yes / OpenAI APIs no", "n/a"),
    ("Guardrails", "yes", "no"),
    ("Prompt caching", "yes", "per model"),
    ("Provisioned Throughput", "yes", "no"),
    ("Batch inference", "yes", "no (use bedrock-runtime)"),
    ("Quotas", "fixed RPM + TPM", "queued fair-share, no RPM"),
    ("CloudWatch namespace", "AWS/Bedrock", "AWS/BedrockMantle"),
    ("Cost attribution", "IAM principal, profiles", "Projects / Workspaces"),
]
w = 28
print(f"{'':{w}} {'bedrock-runtime':32} bedrock-mantle")
print("-" * 100)
for row in comparison:
    print(f"{row[0]:{w}} {row[1]:32} {row[2]}")

served_on_runtime = [name for name, rt, _ in api_rows if rt]
print(f"\n=> bedrock-runtime served {len(served_on_runtime)}/3 of the "
      f"OpenAI- and Anthropic-compatible APIs today: {served_on_runtime}")
print("   Both endpoints are a base-URL change away from an OpenAI SDK codebase.")

                             bedrock-runtime                  bedrock-mantle
----------------------------------------------------------------------------------------------------
URL                          bedrock-runtime.{r}.amazonaws.com bedrock-mantle.{r}.api.aws
AWS recommends               yes, for new applications        fully supported
InvokeModel / Converse       yes                              no
Chat Completions (probed)    yes                              yes
Responses (probed)           yes                              yes
Anthropic Messages (probed)  yes                              yes
OpenAI-compatible path       /openai/v1                       /openai/v1 or /v1, by family
Auth                         SigV4 or Bedrock API key         SigV4 or Bedrock API key
OpenAI SDK drop-in           yes (base_url + key)             yes (base_url + key)
Stateful chat                store + previous_response_id     store + previous_response_id
Server-side tools            no        

**Rule of thumb, as AWS states it.** New applications → **`bedrock-runtime`**.
From the
[endpoints page](https://docs.aws.amazon.com/bedrock/latest/userguide/endpoints.html):
*"For new applications, we recommend the `bedrock-runtime` endpoint."* It is also
where Guardrails, intelligent prompt routing and cross-Region inference live.

Reach for `bedrock-mantle` when you specifically need one of the things only it
has today:

- **server-side or pre-configured tool use**, including web search
- **asynchronous inference** (`background=true`) for long-running work
- **Projects or Workspaces**, to isolate workloads and attribute cost per
  application
- **a model that is only on mantle** — Gemma 4, GPT-5.4/5.5, Grok 4.3 and
  DeepSeek v3.1 are in that set today; §8 enumerates it live

If you already run `bedrock-mantle`, nothing is being taken away: AWS's wording
is that existing applications *"continue to be fully supported and do not need to
change."* And both endpoints can be used from the same application — pick per use
case, not once per project.

The collection reflects this: notebooks lead with whichever endpoint is right for
the model in front of them, and say which one they chose and why.

## 2. URL paths — which depend on the endpoint, not just the model

This is the part that trips people up, and it got one level harder in August
2026: the path depends on **both** the model family and the endpoint.

On **`bedrock-mantle`**, three families:

| Path | Families |
|---|---|
| `/openai/v1/…` | `google.gemma-4*`, `openai.gpt-5*`, `xai.*` |
| `/v1/…` | `openai.gpt-oss*` and every Chat-Completions-only family |
| `/anthropic/v1/…` | `anthropic.*` only |

On **`bedrock-runtime`**, two — and the split falls somewhere else:

| Path | Families |
|---|---|
| `/openai/v1/…` | **every** OpenAI-compatible model, `gpt-oss` and `qwen` included |
| `/anthropic/v1/…` | `anthropic.*` only |

There is **no `/v1` inference surface on `bedrock-runtime` at all**. So the same
model moves paths when you move endpoint: `openai.gpt-oss-20b` is `/v1` on mantle,
and its runtime twin `openai.gpt-oss-20b-1:0` is `/openai/v1`.

Control-plane paths (models, files, projects, fine-tuning, data retention) are
mantle's, and always under `/v1/…`, never `/openai/v1/…`.

In [3]:
def api_prefix(model_id: str, endpoint: str = "mantle") -> str:
    """Which URL prefix serves this model's inference APIs, on this endpoint?

    Copy this with the `endpoint` argument. A version that takes only the model ID
    can only be right about one endpoint, and this collection shipped exactly that
    until bedrock-runtime grew the OpenAI-compatible paths.

    Strip the geo/global prefix first. `us.anthropic.claude-opus-5` does not start
    with "anthropic.", so a naive check routes it to /openai/v1 and you get a 404
    that looks like the model is missing. That bug was in this cell while the
    notebook was being written.
    """
    bare = re.sub(r"^(us|eu|apac|global|in)\.", "", model_id)
    if bare.startswith("anthropic."):
        return "/anthropic/v1"
    if endpoint == "runtime":
        # Runtime puts every OpenAI-compatible model on /openai/v1.
        return "/openai/v1"
    if any(bare.startswith(p) for p in ("google.gemma-4", "openai.gpt-5", "xai.")):
        return "/openai/v1"
    return "/v1"


RUNTIME = f"https://bedrock-runtime.{REGION}.amazonaws.com"

print(f"{'model':28} {'mantle':13} runtime")
print("-" * 60)
for m in [
    "google.gemma-4-31b",
    "openai.gpt-5.6-sol",
    "openai.gpt-oss-120b",
    "xai.grok-4.3",
    "anthropic.claude-sonnet-5",
    "qwen.qwen3-32b",
]:
    print(f"{m:28} {api_prefix(m):13} {api_prefix(m, 'runtime')}")

moved = [
    m
    for m in ("openai.gpt-oss-120b", "qwen.qwen3-32b", "google.gemma-4-31b")
    if api_prefix(m) != api_prefix(m, "runtime")
]
print(f"\n=> {len(moved)} of those change path between endpoints: {moved}")
print("   Full URLs for one of them:")
print(f"     mantle : {HOST}{api_prefix('openai.gpt-oss-120b')}/chat/completions")
print(f"     runtime: {RUNTIME}{api_prefix('openai.gpt-oss-120b', 'runtime')}/chat/completions")

model                        mantle        runtime
------------------------------------------------------------
google.gemma-4-31b           /openai/v1    /openai/v1
openai.gpt-5.6-sol           /openai/v1    /openai/v1
openai.gpt-oss-120b          /v1           /openai/v1
xai.grok-4.3                 /openai/v1    /openai/v1
anthropic.claude-sonnet-5    /anthropic/v1 /anthropic/v1
qwen.qwen3-32b               /v1           /openai/v1

=> 2 of those change path between endpoints: ['openai.gpt-oss-120b', 'qwen.qwen3-32b']
   Full URLs for one of them:
     mantle : https://bedrock-mantle.us-east-1.api.aws/v1/chat/completions
     runtime: https://bedrock-runtime.us-east-1.amazonaws.com/openai/v1/chat/completions


## 2b. The 200 that means failure

Get the path wrong on `bedrock-mantle` and you get a 404 or a 400 with a message.
Get it wrong on `bedrock-runtime` and you can get **HTTP 200** — with this in the
body:

```json
{"Output":{"__type":"com.amazon.coral.service#UnknownOperationException"},"Version":"1.0"}
```

That is a routing failure wearing a success code. Any client shaped like

```python
if response.status_code == 200:
    answer = response.json()["choices"][0]["message"]["content"]   # KeyError
```

fails on the *parse*, several frames away from the actual mistake — a wrong URL.

This is worth knowing because it is easy to hit. The Chat Completions page in the
Amazon Bedrock User Guide shows a `bedrock-runtime` base URL of
`https://bedrock-runtime.{region}.amazonaws.com/v1` — without `/openai`. The
endpoints page is the one to trust: *"The OpenAI-compatible APIs are called on the
`/openai/v1` paths of this endpoint."* The cell below sends both so you can see
the difference rather than take our word for it.

**The rule:** on `bedrock-runtime`, treat a body containing `UnknownOperation` as
a failure regardless of status. `_shared/bedrock.py` exposes `unknown_op(payload)`
and `ok(status, payload)` for exactly this.

In [4]:
# Send the same request to the documented-but-unserved path and to the served
# one, and classify by BODY rather than by status code.
PROBE_PATHS = [
    ("/v1/chat/completions", "openai.gpt-oss-120b-1:0"),
    ("/openai/v1/chat/completions", "openai.gpt-oss-120b-1:0"),
    ("/v1/responses", "us.openai.gpt-5.6-sol"),
    ("/openai/v1/responses", "us.openai.gpt-5.6-sol"),
    ("/anthropic/v1/messages", "us.anthropic.claude-opus-5"),
]


def classify(path: str, model_id: str) -> tuple[str, str]:
    """Return (status, verdict) for one runtime path, reading the body."""
    if "responses" in path:
        payload = {"model": model_id, "input": "Hi", "max_output_tokens": 16}
    elif "messages" in path:
        payload = {
            "model": model_id,
            "max_tokens": 16,
            "messages": [{"role": "user", "content": "Hi"}],
        }
    else:
        payload = {
            "model": model_id,
            "messages": [{"role": "user", "content": "Hi"}],
            "max_completion_tokens": 16,
        }
    headers = {
        "Authorization": f"Bearer {provide_token(region=REGION)}",
        "Content-Type": "application/json",
    }
    if "messages" in path:
        headers["anthropic-version"] = "2023-06-01"
    req = urllib.request.Request(
        RUNTIME + path,
        data=json.dumps(payload).encode(),
        headers=headers,
        method="POST",
    )
    try:
        # URL is a literal host constant plus a literal path, so the scheme is
        # https by construction and urllib's other schemes are unreachable.
        # nosemgrep: dynamic-urllib-use-detected
        with urllib.request.urlopen(req, timeout=120) as resp:  # nosec B310  # noqa: S310
            status, body = resp.status, resp.read().decode("utf-8", "replace")
    except urllib.error.HTTPError as exc:
        status, body = exc.code, exc.read().decode("utf-8", "replace")
    except Exception as exc:
        return "-", f"{type(exc).__name__}"

    if "UnknownOperation" in body:
        return str(status), "NO SUCH PATH (UnknownOperationException)"
    if status == 200:
        return str(status), "answered"
    try:
        message = json.loads(body).get("error", {}).get("message", "")
    except json.JSONDecodeError:
        message = body[:60]
    return str(status), message[:58]


print(f"{'path':32} {'HTTP':>5}  verdict")
print("-" * 92)
trap_hits = []
for path, model_id in PROBE_PATHS:
    status, verdict = classify(path, model_id)
    if "UnknownOperation" in verdict:
        trap_hits.append(path)
    print(f"{path:32} {status:>5}  {verdict}")

print()
if trap_hits:
    print(f"=> {len(trap_hits)} path(s) returned UnknownOperationException: {trap_hits}")
    codes = {classify(p, m)[0] for p, m in PROBE_PATHS if p in trap_hits}
    print(f"   ...with HTTP status {sorted(codes)}. A status-only check calls that")
    print("   a success. Read the body.")
else:
    print("=> no path returned UnknownOperationException today; the trap may be closed,")
    print("   but keep checking the body — it costs one `in` test.")

path                              HTTP  verdict
--------------------------------------------------------------------------------------------


/v1/chat/completions               200  NO SUCH PATH (UnknownOperationException)


/openai/v1/chat/completions        200  answered


/v1/responses                      200  NO SUCH PATH (UnknownOperationException)


/openai/v1/responses               200  answered


/anthropic/v1/messages             200  answered

=> 2 path(s) returned UnknownOperationException: ['/v1/chat/completions', '/v1/responses']


   ...with HTTP status ['200']. A status-only check calls that
   a success. Read the body.


Three things to take from that table:

1. `/openai/v1/...` is the served path on `bedrock-runtime`. `/v1/...` is not.
2. The wrong path returns **200**, and the right path can return **404** — when
   the model is real but does not serve that API. Status code alone tells you
   almost nothing here; the body tells you everything.
3. A 404 naming the *model* (`The model doesn't exist or doesn't support this
   API`) is a different problem from a 200 naming the *operation*. The first means
   "wrong model for this path", the second means "there is no such path".

## 3. Auth A — SigV4 with the AWS SDK (no API key)

Best for Lambda / ECS / EC2 where a role is already attached: nothing to store,
nothing to rotate. Note two things:

- The **signing name is `bedrock`** (`bedrock-mantle` also works).
- You sign a plain HTTPS request. There is no `boto3.client("bedrock-mantle")`.

In [5]:
print("boto3 services containing 'bedrock':")
print(" ", [s for s in boto3.Session().get_available_services() if "bedrock" in s])
print("\nNote the absence of 'bedrock-mantle' — hence manual signing below.")

boto3 services containing 'bedrock':
  ['bedrock', 'bedrock-agent', 'bedrock-agent-runtime', 'bedrock-agentcore', 'bedrock-agentcore-control', 'bedrock-data-automation', 'bedrock-data-automation-runtime', 'bedrock-runtime']

Note the absence of 'bedrock-mantle' — hence manual signing below.


In [6]:
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest
from botocore.httpsession import URLLib3Session


def sigv4_post(path: str, payload: dict, region: str = REGION, timeout: int = 120):
    """POST to bedrock-mantle signed with ambient AWS credentials."""
    body = json.dumps(payload)
    # get_frozen_credentials() pins a consistent access-key/secret/token triple.
    creds = boto3.Session().get_credentials().get_frozen_credentials()
    req = AWSRequest(
        method="POST",
        url=f"https://bedrock-mantle.{region}.api.aws{path}",
        data=body,
        headers={"Content-Type": "application/json"},
    )
    SigV4Auth(creds, "bedrock", region).add_auth(req)  # signing name = "bedrock"
    resp = URLLib3Session(timeout=timeout).send(req.prepare())
    parsed = json.loads(resp.text) if resp.text.strip() else {}
    return resp.status_code, parsed


status, data = sigv4_post(
    "/openai/v1/responses",
    {
        "model": "google.gemma-4-31b",
        "input": "Reply with exactly: OK",
        "max_output_tokens": 16,
    },
)
# The wire format has NO top-level "output_text" -- that is a convenience the
# OpenAI SDK computes. On the raw body the answer is at
# output[].content[].text, which is what response_text() walks.
print("SigV4 ->", status, "|", repr(response_text(data)[:60]))
print("top-level keys:", sorted(data)[:6], "...")
print("output_text present on the wire?", "output_text" in data)

SigV4 -> 200 | 'OK'
top-level keys: ['background', 'billing', 'completed_at', 'created_at', 'error', 'frequency_penalty'] ...
output_text present on the wire? False


## 4. Auth B — short-term Bedrock API key (what the OpenAI SDK needs)

The OpenAI SDK can only send a bearer token; it cannot SigV4-sign. AWS's
official `aws-bedrock-token-generator` mints a **short-term** key from your
normal IAM credentials.

Key facts from the library's own docs:
- It is a **pre-signed SigV4 request** in a wrapper — so it *is* IAM.
- Max lifetime **12 hours**; actual = min(requested, credential expiry).
- **Cannot be refreshed or extended** — you mint a new one.
- It is **Region-pinned** to the Region it was minted in.

Avoid *long-term* keys outside exploration: they create a real IAM user with a
static credential.

In [7]:
from aws_bedrock_token_generator import provide_token

api_key = provide_token(region=REGION)
print("token prefix:", api_key[:22] + "…")
print("length:", len(api_key))
print("\nThe prefix identifies it as a presigned-URL bearer token:")
print(" ", api_key.split("-")[0:3])

token prefix: bedrock-api-key-YmVkcm…
length: 460

The prefix identifies it as a presigned-URL bearer token:
  ['bedrock', 'api', 'key']


### A self-refreshing provider

Because tokens cannot be refreshed, long-lived processes should mint on demand
and track expiry themselves. Keep the TTL short (15 min is plenty) — the token
*is* your role until it expires.

In [8]:
import threading
from datetime import datetime, timedelta, timezone


class MantleTokenProvider:
    """Mints short-term Bedrock tokens, refreshing before expiry."""

    def __init__(self, region=REGION, ttl=timedelta(minutes=15), skew_s=120):
        self.region, self.ttl, self.skew_s = region, ttl, skew_s
        self._token = None
        self._expires_at = None
        self._lock = threading.Lock()

    def get(self) -> str:
        now = datetime.now(timezone.utc)
        with self._lock:  # avoid a thundering herd of mints
            if self._token and self._expires_at and now < self._expires_at:
                return self._token
            self._token = provide_token(region=self.region, expiry=self.ttl)
            self._expires_at = now + self.ttl - timedelta(seconds=self.skew_s)
            return self._token


tokens = MantleTokenProvider()
t1 = tokens.get()
t2 = tokens.get()  # served from cache
print("cached on second call:", t1 == t2)
print("expires around:", tokens._expires_at.isoformat(timespec="seconds"))

cached on second call: True
expires around: 2026-08-22T04:26:32+00:00


### The OpenAI SDK, pointed at Bedrock

Note: build the client from a *fresh* token. Don't construct one at import time
and reuse it for hours — the baked-in key expires.

In [9]:
from openai import OpenAI

gemma = OpenAI(api_key=tokens.get(), base_url=HOST + "/openai/v1")
r = gemma.responses.create(
    model="google.gemma-4-31b",
    input="Reply with exactly: OK",
    max_output_tokens=16,
)
print("OpenAI SDK ->", repr(r.output_text))

OpenAI SDK -> 'OK'


## 5. Auth C — curl

Both auth styles work from the shell. Useful for smoke tests and CI.

**Pass the key through the environment, never on the command line.** Process
arguments are world-readable via `ps` on most systems, so a key interpolated
into an argument leaks to every local user (CWE-214, *Invocation of Process
Using Visible Sensitive Information*). `curl` reads `$BEDROCK_API_KEY` from the
environment it inherits, so the literal never appears in the argument list.

In [10]:
import subprocess  # nosec B404  # noqa: S404

CURL_BEARER = f"""curl -sS -X POST "{HOST}/openai/v1/responses" \
  -H "Authorization: Bearer $BEDROCK_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{{"model":"google.gemma-4-31b","input":"Reply OK","max_output_tokens":16}}'"""
print("Bearer-token form:\n", CURL_BEARER, "\n")

# The command string is a fixed literal defined above -- no untrusted input is
# interpolated -- and the secret arrives via `env`, not argv.
# Absolute interpreter path (never resolved via $PATH) and a fixed argv built
# from the literal above -- no untrusted input reaches the command line.
completed = subprocess.run(  # nosec B603 B607  # noqa: S603
    ["/bin/bash", "-c", CURL_BEARER],
    capture_output=True,
    text=True,
    timeout=180,
    check=False,
    env={**os.environ, "BEDROCK_API_KEY": api_key},
)
if completed.returncode != 0:
    print("curl failed:", completed.stderr[:200])
else:
    body = json.loads(completed.stdout)
    # Same trap as section 3: read output[].content[].text, not "output_text".
    print("live result:", repr(response_text(body)[:80]))

Bearer-token form:
 curl -sS -X POST "https://bedrock-mantle.us-east-1.api.aws/openai/v1/responses"   -H "Authorization: Bearer $BEDROCK_API_KEY"   -H "Content-Type: application/json"   -d '{"model":"google.gemma-4-31b","input":"Reply OK","max_output_tokens":16}' 



live result: 'OK'


In [11]:
# curl can also SigV4-sign natively (curl >= 7.75). Signing name is "bedrock".
curl_sigv4 = f"""curl -sS -X POST "{HOST}/openai/v1/responses" \
  -H "Content-Type: application/json" \
  --aws-sigv4 "aws:amz:{REGION}:bedrock" \
  --user "$AWS_ACCESS_KEY_ID:$AWS_SECRET_ACCESS_KEY" \
  -H "x-amz-security-token: $AWS_SESSION_TOKEN" \
  -d '{{"model":"google.gemma-4-31b","input":"Reply OK","max_output_tokens":16}}'"""
print("SigV4 form (note: needs x-amz-security-token when using temporary creds):")
print(curl_sigv4)

SigV4 form (note: needs x-amz-security-token when using temporary creds):
curl -sS -X POST "https://bedrock-mantle.us-east-1.api.aws/openai/v1/responses"   -H "Content-Type: application/json"   --aws-sigv4 "aws:amz:us-east-1:bedrock"   --user "$AWS_ACCESS_KEY_ID:$AWS_SECRET_ACCESS_KEY"   -H "x-amz-security-token: $AWS_SESSION_TOKEN"   -d '{"model":"google.gemma-4-31b","input":"Reply OK","max_output_tokens":16}'


## 6. Model discovery

`GET /v1/models` is the authoritative inventory. **`/openai/v1/models` returns
404** — the control plane lives under `/v1` only.

In [12]:
def open_https(req, timeout: int):
    """urlopen restricted to HTTPS.

    urllib also honours file://, ftp:// and data:// . These URLs are all built
    from literals, but a client that ever takes a URL from data would let those
    schemes read local files, so the guard belongs in the helper (CWE-22).
    """
    if not req.full_url.startswith("https://"):
        raise ValueError(f"refusing non-HTTPS URL: {req.full_url[:60]}")
    # nosemgrep: dynamic-urllib-use-detected - scheme verified https above
    return urllib.request.urlopen(req, timeout=timeout)  # nosec B310  # noqa: S310


def get_json(path: str, region: str = REGION):
    req = urllib.request.Request(
        f"https://bedrock-mantle.{region}.api.aws{path}",
        headers={"Authorization": f"Bearer {provide_token(region=region)}"},
    )
    try:
        with open_https(req, timeout=90) as resp:
            return resp.status, json.loads(resp.read())
    except urllib.error.HTTPError as e:
        return e.code, {"body": e.read().decode()[:120]}


for path in ("/v1/models", "/openai/v1/models"):
    code, payload = get_json(path)
    n = len(payload.get("data", [])) if code == 200 else "-"
    print(f"GET {path:20} -> {code}  models={n}")

GET /v1/models           -> 200  models=55


GET /openai/v1/models    -> 404  models=-


In [13]:
code, payload = get_json("/v1/models")
model_ids = sorted(m["id"] for m in payload["data"])

families = {}
for mid in model_ids:
    families.setdefault(mid.split(".")[0], []).append(mid)

print(f"{len(model_ids)} models across {len(families)} families in {REGION}\n")
for fam in sorted(families):
    print(f"{fam:12} ({len(families[fam])})")
    for mid in families[fam]:
        print(f"             {mid}")

55 models across 12 families in us-east-1

anthropic    (6)
             anthropic.claude-fable-5
             anthropic.claude-haiku-4-5
             anthropic.claude-opus-4-7
             anthropic.claude-opus-4-8
             anthropic.claude-opus-5
             anthropic.claude-sonnet-5
deepseek     (2)
             deepseek.v3.1
             deepseek.v3.2
google       (6)
             google.gemma-3-12b-it
             google.gemma-3-27b-it
             google.gemma-3-4b-it
             google.gemma-4-26b-a4b
             google.gemma-4-31b
             google.gemma-4-e2b
minimax      (3)
             minimax.minimax-m2
             minimax.minimax-m2.1
             minimax.minimax-m2.5
mistral      (8)
             mistral.devstral-2-123b
             mistral.magistral-small-2509
             mistral.ministral-3-14b-instruct
             mistral.ministral-3-3b-instruct
             mistral.ministral-3-8b-instruct
             mistral.mistral-large-3-675b-instruct
             mis

## 7. Which API does each family support — on which endpoint?

Don't assume; probe. And probe **per endpoint**, because the answer differs. On
`bedrock-mantle` the Responses API reaches a minority of families and Claude is
Messages-only. On `bedrock-runtime` the shape is different again: Chat Completions
is broad, Responses is narrow, and Messages serves only the newest Claude models.

One more wrinkle before the code: **the model ID is not the same on both
endpoints.** The next cell resolves it rather than assuming, because a mantle ID
sent to runtime returns *"The provided model identifier is invalid"* — which reads
like a missing model rather than a missing translation.

In [14]:
from bedrock import runtime_id_for  # maps a mantle model ID to its runtime form

AV = {"anthropic-version": "2023-06-01"}


def _post(base: str, path: str, payload: dict, extra: dict | None = None):
    headers = {
        "Authorization": f"Bearer {provide_token(region=REGION)}",
        "Content-Type": "application/json",
    }
    if extra:
        headers.update(extra)
    req = urllib.request.Request(
        base + path, data=json.dumps(payload).encode(), headers=headers, method="POST"
    )
    try:
        # URL is a literal host constant plus a literal path, so the scheme is
        # https by construction and urllib's other schemes are unreachable.
        # nosemgrep: dynamic-urllib-use-detected
        with urllib.request.urlopen(req, timeout=120) as resp:  # nosec B310  # noqa: S310
            return resp.status, resp.read().decode("utf-8", "replace")
    except urllib.error.HTTPError as exc:
        return exc.code, exc.read().decode("utf-8", "replace")
    except Exception as exc:
        return -1, f"{type(exc).__name__}"


def probe_apis(model_id: str, endpoint: str) -> dict:
    """Status per API for one model on one endpoint. 'ok' means a real 200."""
    base = HOST if endpoint == "mantle" else RUNTIME
    mid = model_id if endpoint == "mantle" else runtime_id_for(model_id, REGION)
    out = {"model": model_id, "id": mid or "-"}
    if mid is None:
        return {**out, "responses": "n/a", "chat": "n/a", "messages": "n/a"}

    prefix = api_prefix(mid, endpoint)

    def verdict(status, body):
        if "UnknownOperation" in body:
            return "no path"
        return "ok" if status == 200 else str(status)

    if prefix == "/anthropic/v1":
        out["responses"] = out["chat"] = "-"
        out["messages"] = verdict(
            *_post(
                base,
                f"{prefix}/messages",
                {
                    "model": mid,
                    "max_tokens": 16,
                    "messages": [{"role": "user", "content": "Hi"}],
                },
                AV,
            )
        )
        return out

    out["messages"] = "-"
    out["responses"] = verdict(
        *_post(base, f"{prefix}/responses",
               {"model": mid, "input": "Hi", "max_output_tokens": 16})
    )
    # Try both budget field names before concluding an API is missing. gpt-5.6
    # serves Chat Completions but refuses `max_tokens`, and reading that 400 as
    # "no Chat Completions" is how a false claim once reached three notebooks.
    for field in ("max_tokens", "max_completion_tokens"):
        status, body = _post(
            base,
            f"{prefix}/chat/completions",
            {"model": mid, "messages": [{"role": "user", "content": "Hi"}], field: 16},
        )
        if status == 200:
            break
    out["chat"] = verdict(status, body)
    return out


REPRESENTATIVES = [
    "google.gemma-4-31b",
    "openai.gpt-5.6-sol",
    "openai.gpt-oss-120b",
    "xai.grok-4.3",
    "xai.grok-4.6",
    "anthropic.claude-opus-5",
    "qwen.qwen3-32b",
    "deepseek.v3.2",
]

for endpoint in ("mantle", "runtime"):
    print(f"=== {endpoint} " + "=" * 74)
    print(f"{'model asked for':26} {'id used there':32} {'Resp':>7} {'Chat':>7} {'Msg':>7}")
    print("-" * 84)
    for m in REPRESENTATIVES:
        row = probe_apis(m, endpoint)
        print(f"{row['model']:26} {row['id']:32} {row['responses']:>7} "
              f"{row['chat']:>7} {row['messages']:>7}")
    print()

renames = [
    (m, runtime_id_for(m, REGION))
    for m in REPRESENTATIVES
    if runtime_id_for(m, REGION) not in (None, m)
]
missing = [m for m in REPRESENTATIVES if runtime_id_for(m, REGION) is None]
print(f"=> {len(renames)} of {len(REPRESENTATIVES)} models are addressed by a "
      f"DIFFERENT id on runtime:")
for old, new in renames:
    print(f"     {old:26} -> {new}")
print(f"=> {len(missing)} are not on runtime at all: {missing}")

=== mantle ==========================================================================
model asked for            id used there                       Resp    Chat     Msg
------------------------------------------------------------------------------------


google.gemma-4-31b         google.gemma-4-31b                    ok      ok       -


openai.gpt-5.6-sol         openai.gpt-5.6-sol                    ok      ok       -


openai.gpt-oss-120b        openai.gpt-oss-120b                   ok      ok       -


xai.grok-4.3               xai.grok-4.3                          ok      ok       -


...

openai.gpt-oss-120b        openai.gpt-oss-120b-1:0              404      ok       -
xai.grok-4.3               -                                    n/a     n/a     n/a


xai.grok-4.6               us.xai.grok-4.6                       ok      ok       -


anthropic.claude-opus-5    us.anthropic.claude-opus-5             -       -      ok


qwen.qwen3-32b             qwen.qwen3-32b-v1:0                  404      ok       -


deepseek.v3.2              deepseek.v3.2                        400      ok       -

=> 5 of 8 models are addressed by a DIFFERENT id on runtime:
     openai.gpt-5.6-sol         -> us.openai.gpt-5.6-sol
     openai.gpt-oss-120b        -> openai.gpt-oss-120b-1:0
     xai.grok-4.6               -> us.xai.grok-4.6
     anthropic.claude-opus-5    -> us.anthropic.claude-opus-5
     qwen.qwen3-32b             -> qwen.qwen3-32b-v1:0
=> 2 are not on runtime at all: ['google.gemma-4-31b', 'xai.grok-4.3']


Read the error, not just the status code. Four different things arrive here and
they need four different fixes:

| What you see | What it means |
|---|---|
| `200` + `UnknownOperation` in the body | wrong **path** — there is no such operation |
| `404 The model doesn't exist or doesn't support this API` | right path, wrong **model** for it |
| `400 The model 'x' does not support the '/openai/v1/responses' API` | the model is real, the **API** is not available for it |
| `400 Unsupported parameter: 'max_tokens' …` | right path, right model, wrong **parameter** |
| `400 The provided model identifier is invalid` | you sent the **other endpoint's ID** |

The last one is new and it is the easy mistake to make: `openai.gpt-oss-120b` is a
perfectly good model ID *on mantle*, and on runtime it is invalid — the runtime ID
is `openai.gpt-oss-120b-1:0`. `runtime_id_for()` translates.

Concretely, from the tables above:

- **Chat Completions is the broad surface on both endpoints.** On mantle it is the
  universal one for open-weight families; on runtime it reaches most of the
  catalogue too.
- **Responses is narrow on runtime** — the GPT-5.6 profiles and Grok 4.6 today —
  and wider on mantle, where gpt-oss and Gemma 4 also serve it.
- **Claude** is Messages-only on both, and on runtime only the newest models
  answer, addressed by a `us.` or `global.` profile.
- **gpt-5.6** serves both Responses and Chat Completions on both endpoints, and
  rejects `max_tokens` in favour of `max_completion_tokens` on each. That 400 is
  easy to misread as the API being missing; `01-openai-gpt/01` §2b separates them.

Each family notebook in this collection leads with whichever endpoint and API
actually works for its models.

## 8. Regional footprint

Model availability differs sharply by Region. Pin your Region per workload.

In [15]:
REGIONS = ["us-east-1", "us-east-2", "us-west-2", "eu-central-1"]
inventory = {}
for reg in REGIONS:
    code, payload = get_json("/v1/models", region=reg)
    inventory[reg] = (
        sorted(m["id"] for m in payload.get("data", [])) if code == 200 else []
    )
    print(f"{reg:14} {len(inventory[reg]):3} models")

us-east-1       55 models


us-east-2       50 models


us-west-2       48 models


eu-central-1    33 models


In [16]:
watch = [
    "anthropic.claude-opus-5",
    "anthropic.claude-haiku-4-5",
    "openai.gpt-5.6-sol",
    "openai.gpt-oss-120b",
    "google.gemma-4-31b",
    "xai.grok-4.3",
    "qwen.qwen3-32b",
]
print(f"{'model':30} " + "  ".join(f"{r:>13}" for r in REGIONS))
print("-" * 90)
for m in watch:
    cells = "  ".join(f"{('yes' if m in inventory[r] else '-'):>13}" for r in REGIONS)
    print(f"{m:30} {cells}")

model                              us-east-1      us-east-2      us-west-2   eu-central-1
------------------------------------------------------------------------------------------
anthropic.claude-opus-5                  yes              -              -              -
anthropic.claude-haiku-4-5               yes            yes            yes              -
openai.gpt-5.6-sol                       yes            yes              -              -
openai.gpt-oss-120b                      yes            yes            yes            yes
google.gemma-4-31b                       yes            yes            yes            yes
xai.grok-4.3                             yes            yes            yes              -
qwen.qwen3-32b                           yes            yes            yes            yes


Read the table above rather than this prose: **Region coverage differs by family,
and it changes.** New Regions light up, models arrive and are retired, and any list
written here is a snapshot of the day it was run.

The durable practice is the one this section demonstrates: before you commit to a
Region, enumerate `GET /v1/models` there and check that the families you depend on
are actually present. The notebooks in this collection all target `us-east-1`
because that is where the widest selection was available when they were written —
not because other Regions are unsuitable.

## 9. IAM — the permissions you need

Attach the managed policy for inference:

```bash
aws iam attach-role-policy --role-name YourRole \
  --policy-arn arn:aws:iam::aws:policy/AmazonBedrockMantleInferenceAccess
```

Or scope it yourself. Note that **SigV4 callers do not need
`CallWithBearerToken`** — only bearer-token (API key) callers do.

> **The policy below is a DEVELOPMENT example.** It uses the service-scoped
> wildcards `bedrock-mantle:Get*` and `bedrock-mantle:List*` for brevity. Do not
> copy it into production: a wildcard grants any future action matching the
> pattern, so a service release can silently widen your permissions. The
> production variant in the next cell lists actions explicitly
> (AWS Well-Architected SEC 3, least privilege).

In [17]:
inference_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "MantleInference",
            "Effect": "Allow",
            "Action": [
                "bedrock-mantle:CreateInference",
                "bedrock-mantle:Get*",
                "bedrock-mantle:List*",
            ],
            "Resource": "arn:aws:bedrock-mantle:*:*:project/*",
        },
        {
            # Only required when authenticating with a Bedrock API key.
            "Sid": "MantleBearerToken",
            "Effect": "Allow",
            "Action": "bedrock-mantle:CallWithBearerToken",
            "Resource": "*",
        },
        {
            # First call in a fresh account auto-subscribes via Marketplace.
            "Sid": "MarketplaceAutoSubscribe",
            "Effect": "Allow",
            "Action": [
                "aws-marketplace:Subscribe",
                "aws-marketplace:ViewSubscriptions",
            ],
            "Resource": "*",
            "Condition": {
                "StringEquals": {"aws:CalledViaLast": "bedrock-mantle.amazonaws.com"}
            },
        },
    ],
}
print(json.dumps(inference_policy, indent=2))

{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Sid": "MantleInference",
      "Effect": "Allow",
      "Action": [
        "bedrock-mantle:CreateInference",
        "bedrock-mantle:Get*",
        "bedrock-mantle:List*"
      ],
      "Resource": "arn:aws:bedrock-mantle:*:*:project/*"
    },
    {
      "Sid": "MantleBearerToken",
      "Effect": "Allow",
      "Action": "bedrock-mantle:CallWithBearerToken",
      "Resource": "*"
    },
    {
      "Sid": "MarketplaceAutoSubscribe",
      "Effect": "Allow",
      "Action": [
        "aws-marketplace:Subscribe",
        "aws-marketplace:ViewSubscriptions"
      ],
      "Resource": "*",
      "Condition": {
        "StringEquals": {
          "aws:CalledViaLast": "bedrock-mantle.amazonaws.com"
        }
      }
    }
  ]
}


### The production variant — explicit actions, no wildcards

Copy **this** one. Every action is named, so a future service release cannot
widen the grant, and a reviewer can see exactly what the workload may do.

In [18]:
production_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "MantleInference",
            "Effect": "Allow",
            "Action": [
                "bedrock-mantle:CreateInference",
                "bedrock-mantle:GetModel",
                "bedrock-mantle:ListModels",
                "bedrock-mantle:GetProject",
                "bedrock-mantle:ListProjects",
            ],
            "Resource": "arn:aws:bedrock-mantle:*:*:project/*",
        },
        {
            # Only required when authenticating with a Bedrock API key.
            # Omit this statement entirely if you sign with SigV4.
            "Sid": "MantleBearerToken",
            "Effect": "Allow",
            "Action": "bedrock-mantle:CallWithBearerToken",
            "Resource": "*",
        },
    ],
}
print(json.dumps(production_policy, indent=2))
print("\nDrop the Marketplace statement once the account is subscribed.")

{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Sid": "MantleInference",
      "Effect": "Allow",
      "Action": [
        "bedrock-mantle:CreateInference",
        "bedrock-mantle:GetModel",
        "bedrock-mantle:ListModels",
        "bedrock-mantle:GetProject",
        "bedrock-mantle:ListProjects"
      ],
      "Resource": "arn:aws:bedrock-mantle:*:*:project/*"
    },
    {
      "Sid": "MantleBearerToken",
      "Effect": "Allow",
      "Action": "bedrock-mantle:CallWithBearerToken",
      "Resource": "*"
    }
  ]
}

Drop the Marketplace statement once the account is subscribed.


### Governance: ban long-term API keys org-wide

Short-term keys are fine (they're presigned SigV4 and expire). Long-term keys
create a static IAM user credential. This SCP allows the former and blocks the
latter:

In [19]:
scp = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "DenyLongTermBedrockKeys",
            "Effect": "Deny",
            "Action": [
                "bedrock-mantle:CallWithBearerToken",
                "bedrock:CallWithBearerToken",
            ],
            "Condition": {
                "StringEquals": {"bedrock-mantle:bearerTokenType": "LONG_TERM"}
            },
            "Resource": "*",
        },
        {
            "Sid": "DenyCreatingLongTermKeys",
            "Effect": "Deny",
            "Action": "iam:CreateServiceSpecificCredential",
            "Condition": {
                "StringEquals": {
                    "iam:ServiceSpecificCredentialServiceName": "bedrock.amazonaws.com"
                }
            },
            "Resource": "*",
        },
    ],
}
print(json.dumps(scp, indent=2))
print(
    "\nNote: deny BOTH CallWithBearerToken actions to fully close off API-key access."
)

{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Sid": "DenyLongTermBedrockKeys",
      "Effect": "Deny",
      "Action": [
        "bedrock-mantle:CallWithBearerToken",
        "bedrock:CallWithBearerToken"
      ],
      "Condition": {
        "StringEquals": {
          "bedrock-mantle:bearerTokenType": "LONG_TERM"
        }
      },
      "Resource": "*"
    },
    {
      "Sid": "DenyCreatingLongTermKeys",
      "Effect": "Deny",
      "Action": "iam:CreateServiceSpecificCredential",
      "Condition": {
        "StringEquals": {
          "iam:ServiceSpecificCredentialServiceName": "bedrock.amazonaws.com"
        }
      },
      "Resource": "*"
    }
  ]
}

Note: deny BOTH CallWithBearerToken actions to fully close off API-key access.


## 10. Shared helper used by the rest of this collection

The other notebooks import `_shared/bedrock.py`, which wraps exactly what we
built above (path resolution, token minting, retrying POST, streaming, TTFT
(time-to-first-token)).

In [20]:
sys.path.insert(0, "../_shared")
import bedrock

print("helper API surface:")
for fn in [
    "api_prefix",
    "base_url",
    "client",
    "anthropic_client",
    "post",
    "stream_lines",
    "err",
    "list_models",
    "families",
    "response_text",
    "function_calls",
    "ttft",
]:
    print("  mantle." + fn)

code, data = bedrock.post(
    "/openai/v1/responses",
    {"model": "google.gemma-4-31b", "input": "Reply OK", "max_output_tokens": 16},
)
print("\nhelper smoke test:", code, repr(bedrock.response_text(data)))

helper API surface:
  mantle.api_prefix
  mantle.base_url
  mantle.client
  mantle.anthropic_client
  mantle.post
  mantle.stream_lines
  mantle.err
  mantle.list_models
  mantle.families
  mantle.response_text
  mantle.function_calls
  mantle.ttft



helper smoke test: 200 'OK'


## Gotchas from this notebook

| Gotcha | Detail |
|---|---|
| No boto3 mantle client | Use the SDK to *sign*; there is no `client("bedrock-mantle")` |
| Signing name | `bedrock` (not `bedrock-mantle`) — both work, `bedrock` is canonical |
| Wrong path | `/openai/v1` vs `/v1` vs `/anthropic/v1` → 404/400 |
| `/openai/v1/models` | 404. Inventory is only at `/v1/models` |
| No `output_text` on the wire | It is an SDK convenience. On a raw body read `output[].content[].text` |
| Token lifetime | Max 12h, **cannot be refreshed**; Region-pinned |
| Long-term keys | Exploration only; they create a static IAM user credential |
| Region drift | Claude ≈ us-east-1 only; eu-central-1 carries 33 of 55 models |
| `CallWithBearerToken` | Needed for API keys, **not** for SigV4 |

## Next
- `02-governance-projects-retention-and-observability.ipynb` — Projects, ZDR (zero
  data retention),
  CloudWatch
- `03-scaling-tiers-and-latency.ipynb` — quotas, retries, service tiers, TTFT
- Then jump to your model family: `../01-openai-gpt/`, `../02-anthropic-claude/`, etc.